#### Определение
TruncatedSVD из scikit-learn — это метод снижения размерности, основанный на усечённом сингулярном разложении матрицы. Название "Truncated" означает, что вычисляется не полное разложение, а только первые k компонент. Обычное SVD декомпозирует матрицу A полностью: 
$$A_{m \times n} = U_{m \times m} \cdot \Sigma_{m \times n} \cdot V^T_{n \times n}$$ что требует $O(mn^2)$ операций

TruncatedSVD ограничивается приближением $$A \approx U_{m \times k} \cdot \Sigma_{k \times k} \cdot V^T_{k \times n}$$ беря только k наибольших сингулярных значений и соответствующие им векторы (строки)

По теореме Эккарта–Янга это наилучшее низкоранговое приближение матрицы в смысле нормы Фробениуса. Среди всех матриц ранга не выше k (любых — не только полученных через SVD) наилучшим приближением к A является именно $A_k$. Никакой другой способ выбрать k направлений не даст меньшей ошибки.

Геометрически: $A_k$ — это проекция A на наилучшее k-мерное подпространство. "Наилучшее" здесь означает то, которое захватывает максимальную дисперсию данных — именно это и делает PCA.

Оказывается, ошибка выражается через собственные значения разложения $\|A - A_k\|_F^2 = \sigma_{k+1}^2 + \ldots + \sigma_r^2$. Поэтому при выборе k нужно найти точку, где $\sigma_{k+1}$ уже мала: всё что левее — сигнал, всё что правее — шум.

В sklearn explained_variance_ratio_[i] $\approx \sigma_i^2 / \sum_j \sigma_j^2$. Это прямое следствие теоремы: доля "энергии" матрицы, объяснённая i-й компонентой, пропорциональна $\sigma_i^2$. Cumulative sum даёт $1 - \|A - A_k\|_F^2 / \|A\|_F^2$ — долю сохранённой энергии при усечении до k компонент.

#### Определение
Теоретически метод находит три объекта: 
- U — левые сингулярные векторы (ортонормальный базис пространства строк),
- $\Sigma = \text{diag}(\sigma_1 \geq \sigma_2 \geq \ldots \geq \sigma_k)$ — сингулярные значения,
- V — правые сингулярные векторы (ортонормальный базис пространства столбцов).

Преобразование объекта x в новое пространство выглядит как $x_{\text{new}} = x \cdot V_k$, то есть transform(X) возвращает $X \cdot V_k^T$ — проекцию на k главных направлений.

#### Практический алгоритм
Практически sklearn использует рандомизированный SVD (алгоритм Halko et al., 2009), а не точный, что даёт сложность $O(mnk)$ вместо $O(mn^2)$. 

Алгоритм:
- строит случайную матрицу $\Omega \in \mathbb{R}^{n \times k}$,
- вычисляет $Y = A\Omega$ — случайную проекцию,
- извлекает из Y ортонормальный базис Q,
- делает SVD маленькой матрицы $B = Q^T A$
- восстанавливает приближённые $U, \Sigma, V$

Параметр n_iter контролирует точность через степенные итерации (по умолчанию n_iter=4). Альтернативный алгоритм arpack — итерационный метод Ланцоша, точный, но медленнее; хорош для небольших матриц или когда нужна высокая точность.

#### Связанные метрики
После fit() доступны следующие атрибуты: components_ 
— матрица $V^T$ размера (k, n_features), то есть правые сингулярные векторы; 
- explained_variance_ — дисперсия, объяснённая каждой компонентой;
- explained_variance_ratio_ — доля от общей дисперсии;
- singular_values_ — сингулярные значения $\sigma_1, \ldots, \sigma_k$;
- n_features_in_ — число признаков входной матрицы.

Важный нюанс: explained_variance_ratio_ считается как дисперсия трансформированных данных, не через $\sigma_i^2 / \sum \sigma_j^2$, поэтому сумма может быть меньше 1 даже если взять все компоненты, когда матрица не центрирована.

#### Оптимальное число компонент
Для определения оптимального числа компонент есть несколько подходов. Главный — анализ объяснённой дисперсии: строим SVD с запасом компонент и находим k, при котором cumsum(explained_variance_ratio_) достигает нужного порога, например 90%. Elbow-метод: строим график сингулярных значений по номеру компоненты и ищем "колено" — точку, где значения резко перестают падать. Лучший метод на практике — downstream-метрика: пробовать k из набора {50, 100, 200, 300} и смотреть на качество конечной задачи (F1, AUC). Для LSA на коротких текстах малого корпуса обычно достаточно 50–100 компонент, для большого корпуса с длинными документами берут 200–500; редко имеет смысл брать больше 500

TruncatedSVD наиболее полезен в нескольких областях. В NLP и текстовых задачах это де-факто стандарт: LSA (Latent Semantic Analysis) — снижение размерности TF-IDF матрицы, устранение синонимии и полисемии в векторах слов, сжатие перед подачей в классификаторы. В рекомендательных системах метод применяется для разложения матрицы user–item с целью нахождения латентных факторов. Для сжатия признаков перед SVM или логрегрессией на очень широких данных, для шумоподавления (малые сингулярные значения соответствуют шуму), а также для визуализации при снижении до 2–3 компонент.

Использовать TruncatedSVD стоит, когда матрица разреженная (TF-IDF, one-hot, счётчики), когда данных много и PCA слишком медленный, когда нужно снизить размерность перед другим алгоритмом, или при работе с текстами. Не стоит применять его, если данные плотные и умещаются в память (лучше PCA), если нужна интерпретируемость компонент, или если данные нелинейно распределены (нужен Kernel PCA, UMAP, t-SNE).

#### Альтернативы
Альтернативные методы снижения размерности делятся на линейные и нелинейные. 

Среди линейных: 
- PCA максимизирует дисперсию через ортогональные компоненты и подходит для плотных данных без выбросов
- LDA максимизирует разделимость классов и используется в задачах классификации
- ICA ищет статистически независимые компоненты и применяется для работы с сигналами и разделения источников (BSS)
- Factor Analysis моделирует данные через латентные факторы плюс шум на каждый признак, популярен в психометрии и при анализе анкет
- Random Projection использует случайную матрицу и сохраняет расстояния по теореме Джонсона–Линденштрауса, хорошо работает на очень больших данных
- NMF аппроксимирует матрицу произведением $A \approx WH$ при ограничении неотрицательности всех элементов, применяется для текстов, изображений и аудио

Среди нелинейных методов (manifold learning): 
- t-SNE сохраняет локальную структуру через KL-дивергенцию и используется почти исключительно для визуализации в 2–3 измерениях;
- UMAP основан на риманновой геометрии и топологических инвариантах, применяется для визуализации и иногда в качестве препроцессинга;
- Isomap строит граф соседей и использует геодезические расстояния;
- LLE аппроксимирует окрестности локально линейными преобразованиями;
- Spectral Embedding работает с собственными векторами Лапласиана графа и хорошо сочетается с кластеризацией;
- Autoencoder сжимает данные через нейросетевое бутылочное горлышко и применяется для изображений, звука, текста;
- VAE добавляет к автоэнкодеру нормальное распределение в латентном пространстве и используется для генерации и интерполяции.

Среди методов матричного разложения: 
- SVD $U \Sigma V^T$ с ортогональными U и V описывает геометрию данных через повороты и масштабирование;
- EVD $Q \Lambda Q^{-1}$ применяется только к квадратным матрицам и находит собственные направления;
- QR-разложение на ортогональную и верхнетреугольную матрицы используется для численной устойчивости и метода наименьших квадратов;
- LU-разложение на нижнюю и верхнюю треугольные матрицы применяется для решения СЛАУ; разложение Холецкого $LL^T$ — быстрый аналог LU для положительно определённых матриц;
- NMF с неотрицательными множителями WH даёт интерпретируемые части-объекты;
- CUR-разложение использует реальные столбцы и строки исходной матрицы для повышения интерпретируемости;
- CP и Tucker — тензорные обобщения SVD для многомерных данных;
- ICA разлагает данные на статистически независимые компоненты для разделения источников.

#### Эквивалентность SVD и PCA
Эквивалентность SVD и PCA можно разобрать строго математически. Пусть дана центрированная матрица данных $X \in \mathbb{R}^{n \times p}$, из каждого столбца которой вычтено среднее. PCA ищет собственные векторы матрицы ковариаций $C = \frac{1}{n-1} X^T X \in \mathbb{R}^{p \times p}$ через eigen-разложение $C = V \Lambda V^T$, где V — матрица собственных векторов (главные направления), $\Lambda = \text{diag}(\lambda_1, \ldots, \lambda_p)$ — собственные значения. Проекция данных: $Z = X V_k$.

Если применить SVD к той же центрированной X: $X = U \Sigma V^T$, то матрица ковариаций раскрывается как $X^T X = V \Sigma U^T U \Sigma V^T = V \Sigma^2 V^T$ — это в точности eigen-разложение $X^T X$. Значит $\lambda_i = \frac{\sigma_i^2}{n-1}$, а $V_{\text{PCA}} = V_{\text{SVD}}$. Проекция через SVD: $Z = X V_k = U \Sigma V^T V_k = U_k \Sigma_k$. Итог: $Z_{\text{PCA}} = X V_k = U_k \Sigma_k = Z_{\text{SVD}}$. Главные направления PCA совпадают с правыми сингулярными векторами SVD; собственные значения ковариационной матрицы равны $\sigma_i^2 / (n-1)$; координаты объектов $X V_k$ и $U_k \Sigma_k$ тождественны. При центрированных данных это один и тот же результат, просто найденный разными путями — EVD матрицы $p \times p$ или SVD матрицы $n \times p$.

#### Sparse VS Dense описания
Причина, по которой PCA применяют к плотным данным, а TruncatedSVD к разреженным, целиком определяется одной операцией — центрированием. Рассмотрим разреженную TF-IDF матрицу размером 100 тысяч документов на 50 тысяч слов с заполненностью 0.1%: ненулевых элементов около 5 миллионов из 5 миллиардов. Операция X - X.mean(axis=0), где среднее по столбцу — плотный вектор, делает все нули ненулевыми. Результат — 5 миллиардов элементов, около 40 ГБ памяти. Разреженность уничтожается полностью. Именно поэтому sklearn.PCA честно отказывается принимать разреженные матрицы с ошибкой типа. TruncatedSVD никогда не центрирует — он работает напрямую с матрицей X через рандомизированный SVD, выполняя только умножение X на случайную матрицу (sparse × dense = dense, эффективно) и QR-разложение маленькой матрицы. Разреженность сохраняется на всём протяжении вычислений.

Дополнительное семантическое следствие отсутствия центрирования: TruncatedSVD захватывает среднее как первую компоненту. На TF-IDF матрице первая компонента часто отражает просто "длину документа" (частые слова везде), а не реальную тематику. В практике LSA это нередко решают, отбрасывая нулевую компоненту и оставляя только с первой по k-1.

Если применить PCA к большой разреженной матрице, sklearn просто упадёт с TypeError. Если форсировать преобразование через .toarray(), процесс будет убит OOM-killer'ом ядра или бросит MemoryError. Если матрица небольшая и влезла в память, PCA технически отработает, но возникает семантическая проблема: центрирование для разреженных данных создаёт шум из нулей. У слова "the" среднее TF-IDF равно 0.003, и вычитание этого числа из 999 нулей создаёт -0.003 везде — это шум, не сигнал. Первая компонента PCA найдёт направление максимальной дисперсии, которое для TF-IDF почти всегда совпадает с "длиной документа".

Если применить TruncatedSVD к плотной нецентрированной матрице, технической ошибки не будет — SVD работает на любой матрице. Проблемы другого рода. Во-первых, первая компонента поглощает среднее: если $\mu = \frac{1}{n}\sum x_i \neq 0$, то первый правый сингулярный вектор $v_1 \approx \frac{\mu}{||\mu||}$, то есть тратится на кодирование константного сдвига, а не структуры. Это легко проверить: косинусное сходство первой компоненты с нормированным вектором среднего окажется близким к единице. Во-вторых, теряются полезные компоненты: если первая компонента "занята" средним, то из k компонент полезны лишь k-1. При малом k это существенно. В-третьих, explained_variance_ratio_ лжёт: на данных с большим сдвигом SVD может сообщить, что первая компонента объясняет 99% дисперсии — это статистически бессмысленная цифра, отражающая лишь расстояние облака точек от начала координат. PCA на тех же данных честно покажет равномерное распределение дисперсии. Правило: плотные данные следует сначала центрировать через StandardScaler, затем передавать в PCA; разреженные данные передаются в TruncatedSVD без предобработки; при желании использовать SVD на плотных данных нужно центрировать вручную — тогда результат совпадёт с PCA.

#### SVD перед кластеризацией / классификацией
TruncatedSVD перед кластеризацией имеет смысл по фундаментальной причине — проклятию размерности. В высокоразмерном пространстве все точки оказываются примерно на одинаковом расстоянии друг от друга: для случайных векторов в $\mathbb{R}^d$ при $d \to \infty$ выполняется $\frac{\max_{\text{dist}} - \min_{\text{dist}}}{\min_{\text{dist}}} \to 0$. Все расстояния концентрируются вокруг одного значения, и K-Means с DBSCAN слепнут. SVD находит k осей, вдоль которых данные максимально варьируются — именно те оси, вдоль которых кластеры разнесены. Шумовые измерения с малыми сингулярными значениями отбрасываются. Конкретный эффект для K-Means: евклидовы расстояния в $\mathbb{R}^k$ становятся информативными; центроиды кластеров устойчивее (меньше измерений — меньше шума при усреднении); сходимость быстрее. Стандартный пайплайн для текстов — TruncatedSVD, затем Normalizer (переводит евклидово расстояние в косинусное, что лучше для текстов), затем KMeans.

Перед end-to-end задачами, такими как классификация, TruncatedSVD выполняет три роли. Первая — регуляризация через сжатие представления: классификатор обучается на k признаках вместо p, при k ≪ p у него физически меньше степеней свободы и он не может запомнить шумовые измерения из обучающей выборки. Логрегрессия на 50 тысячах TF-IDF признаков легко запоминает редкие слова из train; та же логрегрессия на 200 SVD компонентах вынуждена работать с обобщёнными темами. Вторая роль — устранение мультиколлинеарности: TF-IDF признаки сильно коррелированы (синонимы, однокоренные слова), тогда как SVD компоненты ортогональны по построению и их корреляция равна нулю; для линейных моделей это критично. Третья роль — ускорение обучения: SVM на 200 признаках работает в сотни раз быстрее, чем на 50 тысячах, поскольку каждое скалярное произведение дешевле.

Аналогия TruncatedSVD с удалением "высокочастотного шума" частично верна, но требует уточнения. В Фурье-анализе высокие частоты соответствуют быстрым колебаниям и деталям. В SVD малые сингулярные значения соответствуют направлениям с малой дисперсией — редким паттернам, которые действительно часто являются шумом. Отбрасывая компоненты с малыми $\sigma_i$, мы убираем то, что слабо варьируется, и это снижает дисперсию модели (variance). Однако аналогия неточна в нескольких аспектах. Частота в Фурье — объективное понятие, тогда как "малая дисперсия" не тождественна шуму. Малые компоненты могут нести редкий, но важный сигнал: в медицинских данных редкий симптом у 1% пациентов даёт малую дисперсию, но является ключевым предиктором болезни — TruncatedSVD его выбросит. Кроме того, базис Фурье фиксирован (синусы и косинусы), тогда как базис SVD адаптивен к данным. Точная формулировка: TruncatedSVD убирает направления с наименьшей дисперсией в обучающих данных. Это снижает дисперсию модели (variance), но может увеличить смещение (bias), если выброшенные компоненты несли полезный сигнал. Это регуляризация через явное ограничение ранга, а не частотная фильтрация.

#### Полный SVD
Полный (нетрункированный) SVD имеет смысл в нескольких ситуациях. В численных методах и при решении СЛАУ нужна псевдообратная матрица для метода наименьших квадратов — для корректного вычисления pinv требуются все сингулярные значения до самого малого, иначе решение будет неточным. Для определения числового ранга матрицы и числа обусловленности (отношение максимального сингулярного значения к минимальному) также нужен полный спектр. При полном восстановлении матрицы или нахождении минимального k с заданной точностью — например, найти наименьшее k, при котором $\sum_{i=1}^k \sigma_i^2 / \sum_{i=1}^n \sigma_i^2 \geq 0.9999$ — необходимо знать все сингулярные значения заранее. Для whitening (сферизации данных), когда все направления приводятся к единичной дисперсии перед ICA, нужен полный SVD, чтобы не исказить масштаб. В научных вычислениях (физика, квантовая химия, метод конечных элементов) требуется гарантированная точность, и нельзя просто взять k=100 и надеяться на приемлемый результат. Наконец, анализ всего спектра сингулярных значений на предмет spectral gap — резкого падения, которое маркирует границу между сигналом и шумом — невозможен без полного SVD: если явного разрыва нет, структура данных неясна; если есть, его позиция подсказывает оптимальный k для TruncatedSVD.


#### Теорема Эккарта–Янга (1936)

Пусть $A \in \mathbb{R}^{m \times n}$ — произвольная матрица с SVD $A = U\Sigma V^T$, и обозначим через $A_k$ её усечённое приближение:

$$A_k = \sum_{i=1}^{k} \sigma_i u_i v_i^T = U_k \Sigma_k V_k^T$$

Тогда $A_k$ является решением задачи наилучшего низкорангового приближения:

$$A_k = \underset{\text{rank}(B) \leq k}{\arg\min} \|A - B\|_F$$

и одновременно

$$A_k = \underset{\text{rank}(B) \leq k}{\arg\min} \|A - B\|_2$$

Ошибки приближения выражаются явно:

$$\|A - A_k\|_F = \sqrt{\sigma_{k+1}^2 + \sigma_{k+2}^2 + \ldots + \sigma_r^2}$$

$$\|A - A_k\|_2 = \sigma_{k+1}$$

Норма Фробениуса матрицы $A \in \mathbb{R}^{m \times n}$ — это просто корень из суммы квадратов всех элементов: $\|A\|_F = \sqrt{\sum_{i=1}^{m} \sum_{j=1}^{n} a_{ij}^2}$

Это обобщение евклидовой нормы вектора на матрицы — можно думать о матрице как о вектор из $mn$ элементов и посчитать его длину. Через сингулярные значения:

$$\|A\|_F = \sqrt{\sigma_1^2 + \sigma_2^2 + \ldots + \sigma_r^2}$$

где $r = \text{rank}(A)$. Это следует из инвариантности нормы Фробениуса относительно ортогональных преобразований: $\|A\|_F = \|U\Sigma V^T\|_F = \|\Sigma\|_F$.

Норма Фробениуса измеряет "суммарную энергию" матрицы. В контексте SVD она отвечает на вопрос: насколько хорошо низкоранговое приближение восстанавливает исходные данные в среднеквадратичном смысле.

Есть и вторая норма, через которую формулируют теорему — спектральная (операторная) норма:

$$\|A\|_2 = \sigma_1$$

это просто наибольшее сингулярное значение, то есть максимальное растяжение, которое матрица производит с единичным вектором.

#### Рандомизированный алгоритм SVD

Представим что нам нужны первые k=10 компонент матрицы 100000×50000. Точный SVD перебирает всё пространство. Но ключевое наблюдение:

> Если мы случайно "пощупаем" матрицу в нескольких направлениях, с высокой вероятностью попадём в подпространство, где сосредоточена почти вся энергия.

Это контринтуитивно, но работает потому что в типичных матрицах данных спектр убывает быстро — почти всё "содержание" матрицы сосредоточено в нескольких направлениях.

Дана матрица $A \in \mathbb{R}^{m \times n}$, хотим k компонент.

**Шаг 1 — случайная проекция**

Генерируем случайную матрицу $\Omega \in \mathbb{R}^{n \times k}$, элементы которой $\omega_{ij} \sim \mathcal{N}(0, 1)$.

Вычисляем $Y = A\Omega \in \mathbb{R}^{m \times k}$.

Каждый столбец Y — это случайная линейная комбинация столбцов A. Множество этих столбцов приближённо натягивает образ A (image of A).

**Шаг 2 — ортонормальный базис**

Делаем QR-разложение: $Y = QR$, где $Q \in \mathbb{R}^{m \times k}$ — ортонормальный базис, аппроксимирующий пространство столбцов A.

Ключевое свойство: $A \approx QQ^TA$. Матрица $QQ^T$ — это проектор на найденное подпространство.

**Шаг 3 — редукция размера**

Вычисляем $B = Q^T A \in \mathbb{R}^{k \times n}$.

Это маленькая матрица! Вместо $m \times n$ — теперь $k \times n$. Мы спроецировали A в найденное подпространство.

**Шаг 4 — точный SVD маленькой матрицы**

$B = \hat{U} \Sigma V^T$ — это дёшево, потому что B маленькая ($k \times n$, где $k \ll m$).

**Шаг 5 — восстановление**

$U = Q\hat{U}$

Итоговое разложение: $A \approx U \Sigma V^T$

### Сложность

```
Точный SVD:          O(mn · min(m,n))
Рандомизированный:   O(mnk)           — при k ≪ min(m,n)
```

На матрице 100k × 50k с k=100: ускорение в ~500 раз.

### Степенные итерации (параметр n_iter)

Базовый алгоритм плохо работает если спектр убывает медленно — малые и большие сингулярные значения "перемешиваются" в случайной проекции. Трюк: применить матрицу несколько раз перед проекцией:

$$Y = (AA^T)^q A\Omega$$

Почему это помогает? Если $A$ имеет сингулярные значения $\sigma_1 \geq \sigma_2 \geq \ldots$, то $(AA^T)^q$ имеет значения $\sigma_1^{2q} \geq \sigma_2^{2q} \geq \ldots$. Возведение в степень растягивает разрыв между значениями:

```
q=0:  σ₁=10, σ₂=8, σ₃=1   — плохо разделены
q=2:  σ₁=10000, σ₂=4096, σ₃=1   — хорошо разделены
```

Каждая итерация умножает на A и $A^T$ — два прохода по данным. При n_iter=4 (по умолчанию) делается 4 таких цикла. Точность растёт, скорость падает линейно.

### Почему случайные гауссовы векторы

Случайный вектор $\omega \sim \mathcal{N}(0, I)$ имеет ненулевую проекцию на любое фиксированное направление с вероятностью 1. Более того, по концентрации меры в высоких размерностях k случайных гауссовых векторов почти ортогональны между собой и почти наверняка "накрывают" k-мерное подпространство с большой энергией. Это гарантирует что $Q$ хорошо аппроксимирует нужное подпространство.

Формально: с вероятностью не менее $1 - \delta$,

$$\|A - QQ^TA\|_2 \leq \left(1 + \frac{k}{\sqrt{\delta}}\right) \sigma_{k+1}$$

то есть ошибка приближения близка к минимально возможной $\sigma_{k+1}$ (что даёт точный truncated SVD по теореме Эккарта–Янга).

---

## Метод Ланцоша

### Общая идея: проекция Крылова

Метод Ланцоша — это итерационный алгоритм для нахождения крайних собственных значений и векторов симметричной матрицы. Для SVD он применяется к $A^TA$ или $AA^T$.

Центральная идея — вместо работы со всей матрицей $A^TA \in \mathbb{R}^{n \times n}$ строится маленькое **крыловское подпространство**:

$$\mathcal{K}_j(A, v) = \text{span}\{v, Av, A^2v, \ldots, A^{j-1}v\}$$

Это подпространство, порождённое последовательными применениями матрицы к одному вектору. Звучит просто, но у этой последовательности есть замечательное свойство: она быстро "стягивается" к собственному вектору, соответствующему наибольшему собственному значению.

### Алгоритм

Дана симметричная матрица $M = A^TA \in \mathbb{R}^{n \times n}$ (для SVD). Стартовый вектор $v_1$ — случайный нормированный.

На каждом шаге j строим новый вектор крыловского подпространства и ортогонализируем его по отношению ко всем предыдущим (процесс Грама–Шмидта):

$$\beta_j v_{j+1} = M v_j - \alpha_j v_j - \beta_{j-1} v_{j-1}$$

где $\alpha_j = v_j^T M v_j$ — диагональные элементы, $\beta_j = \|M v_j - \alpha_j v_j - \beta_{j-1} v_{j-1}\|$ — внедиагональные.

После j шагов матрица M в базисе $\{v_1, \ldots, v_j\}$ имеет трёхдиагональную форму:

$$T_j = \begin{pmatrix} \alpha_1 & \beta_1 & & \\ \beta_1 & \alpha_2 & \beta_2 & \\ & \beta_2 & \ddots & \beta_{j-1} \\ & & \beta_{j-1} & \alpha_j \end{pmatrix}$$

Собственные значения $T_j$ (риц-значения) быстро сходятся к крайним собственным значениям M при $j \ll n$.

### Почему трёхдиагональная матрица

Это следствие симметрии M и ортогональности крыловских векторов. Из рекурсии видно что $v_{j+1}$ ортогонален ко всем предыдущим кроме $v_j$ и $v_{j-1}$ — следовательно матрица в этом базисе имеет только три ненулевые диагонали.

### Сходимость: почему это работает

Последовательность $v, Mv, M^2v, \ldots$ сходится к собственному вектору при наибольшем $|\lambda|$. Крыловское подпространство за j шагов содержит информацию о j-м многочлене от M, применённом к v. Теория полиномиальной аппроксимации показывает: для нахождения k крайних собственных значений с точностью $\varepsilon$ нужно порядка $k + O(\log(1/\varepsilon))$ итераций — намного меньше, чем размерность матрицы n.

### Практические проблемы

В теории Ланцош делает ровно n шагов и находит все собственные значения. На практике из-за ошибок округления ортогональность между векторами теряется, и начинают появляться **фантомные собственные значения** (ghost eigenvalues) — ложные копии уже найденных. Решение — явная повторная ортогонализация (reorthogonalization) на каждом шаге, что увеличивает стоимость до $O(jn)$ за j шагов, но гарантирует корректность.

### Ланцош против рандомизированного SVD

| | Ланцош (arpack) | Рандомизированный |
|---|---|---|
| Тип | Детерминированный итерационный | Вероятностный одно-проходный |
| Точность | Высокая, контролируемая | Зависит от n_iter |
| Скорость | Медленнее при больших k | Быстрее при больших k |
| Медленно убывающий спектр | Справляется хорошо | Нужны итерации |
| Реализация | ARPACK (Fortran, 30 лет) | Halko et al. 2009 |
| Когда брать | Малые данные, нужна точность | Большие данные, нужна скорость |

---

